In [349]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot  as plt
car=pd.read_csv("quikr_car.csv")

In [350]:
car=car[car['year'].str.isnumeric()]
car['year']=car['year'].astype(int)
car.info()

<class 'pandas.core.frame.DataFrame'>
Index: 842 entries, 0 to 891
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        842 non-null    object
 1   company     842 non-null    object
 2   year        842 non-null    int64 
 3   Price       842 non-null    object
 4   kms_driven  840 non-null    object
 5   fuel_type   837 non-null    object
dtypes: int64(1), object(5)
memory usage: 46.0+ KB


In [351]:
car=car[car["Price"]!="Ask For Price"]
car["Price"]=car["Price"].str.replace(",","").astype(int)

In [352]:
car["kms_driven"]=car["kms_driven"].str.replace(" kms","")

car["kms_driven"]=car["kms_driven"].str.replace(",","")

car=car[car["kms_driven"].str.isnumeric()]

car["kms_driven"] = car["kms_driven"].astype(int)


In [353]:
car=car[~car["fuel_type"].isna()]

In [354]:
car.info()

<class 'pandas.core.frame.DataFrame'>
Index: 816 entries, 0 to 889
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        816 non-null    object
 1   company     816 non-null    object
 2   year        816 non-null    int64 
 3   Price       816 non-null    int64 
 4   kms_driven  816 non-null    int64 
 5   fuel_type   816 non-null    object
dtypes: int64(3), object(3)
memory usage: 44.6+ KB


In [355]:
car["name"]=car["name"].str.split(" ").str.slice(0,3).str.join(" ")
car.reset_index(drop=True)

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing,Hyundai,2007,80000,45000,Petrol
1,Mahindra Jeep CL550,Mahindra,2006,425000,40,Diesel
2,Hyundai Grand i10,Hyundai,2014,325000,28000,Petrol
3,Ford EcoSport Titanium,Ford,2014,575000,36000,Diesel
4,Ford Figo,Ford,2012,175000,41000,Diesel
...,...,...,...,...,...,...
811,Maruti Suzuki Ritz,Maruti,2011,270000,50000,Petrol
812,Tata Indica V2,Tata,2009,110000,30000,Diesel
813,Toyota Corolla Altis,Toyota,2009,300000,132000,Petrol
814,Tata Zest XM,Tata,2018,260000,27000,Diesel


In [356]:
car=car[car["Price"]<6e6].reset_index(drop=True)

In [357]:
car.to_csv("cleanedData.csv")

In [358]:
x=car.drop("Price",axis=1)
y=car["Price"]

In [359]:
from sklearn.model_selection import train_test_split

xtrain,xtest,ytrain,ytest=train_test_split(x,y,random_state=4,test_size=0.2)

In [360]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import  LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
ohe=OneHotEncoder()

ohe.fit(x[["name","company","fuel_type"]])

colTransform=make_column_transformer((OneHotEncoder(categories=ohe.categories_),["name","company","fuel_type"]),remainder="passthrough")

In [361]:
lr=LinearRegression()

In [362]:
pipe=make_pipeline(colTransform,lr)

pipe.fit(xtrain,ytrain)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(categories=[array(['Audi A3 Cabriolet', 'Audi A4 1.8', 'Audi A4 2.0', 'Audi A6 2.0',
       'Audi A8', 'Audi Q3 2.0', 'Audi Q5 2.0', 'Audi Q7', 'BMW 3 Series',
       'BMW 5 Series', 'BMW 7 Series', 'BMW X1', 'BMW X1 sDrive20d',
       'BMW X1 xDrive20d', 'Chevrolet Beat', 'Chevrolet Beat...
                                                                            array(['Audi', 'BMW', 'Chevrolet', 'Datsun', 'Fiat', 'Force', 'Ford',
       'Hindustan', 'Honda', 'Hyundai', 'Jaguar', 'Jeep', 'Land',
       'Mahindra', 'Maruti', 'Mercedes', 'Mini', 'Mitsubishi', 'Nissan',
       'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo'],
      dtype=object),
                                                                            array(['Diesel', 'LPG', 'Petrol'], dtype=object)]),
                                                  ['name', 'company',
                                                   'fuel_type'])])),
                ('linearregression', LinearRegression())])

In [363]:
pipe.score(xtest,ytest)

0.8189003209843422

In [364]:
print(ytest)
print(pipe.predict(xtest))


477    120000
214    275000
811    110000
705    215000
794    290000
        ...  
436    500000
228    320000
594    550000
364    350000
772     85000
Name: Price, Length: 163, dtype: int64
[  54152.19624361  498250.13247566  230638.7996235   232054.9136032
  430970.18438525  442794.05505741  172270.45443971  393472.26594341
  432960.25979673  346955.23213653  434028.26233327  293499.94138963
  331005.46411523 -106997.18443661  282440.01311839  635899.18495935
  230409.25460007  555585.58076548  273727.32936998  245827.61500877
  175510.08793921  246620.16588171  344303.529652    353227.03059451
  104843.3938295   420215.78668814  134253.83305529  914847.81396832
  297692.04376956 1198889.78459629  348817.04343676  353613.49868207
  631882.1470494   284000.60205994  560662.06472103  282509.60978922
  313497.53254056  560796.06264996  307942.07882328   88971.4056437
  371298.42070322  327255.30161293  717139.21992509 1540082.95907569
  246592.76508686  420053.84912769  286471.5915918

In [365]:
from sklearn.metrics import r2_score

r2_score(ytest,pipe.predict(xtest))

0.8189003209843422

In [366]:
scores=[]
for i in range(30):
    xtrain,xtest,ytrain,ytest=train_test_split(x,y,random_state=i,test_size=0.2)
    lr=LinearRegression()
    pipe=make_pipeline(colTransform,lr)
    pipe.fit(xtrain,ytrain)
    scores.append(r2_score(ytest,pipe.predict(xtest)))

scores    
    

[0.6589775490516477,
 0.47859534625114053,
 0.6206184325887315,
 0.5370851059720391,
 0.8189003209843422,
 0.6634592678218548,
 0.627651080774843,
 0.6298500612485975,
 0.6721337147755825,
 0.569999290976279,
 0.648410356051369,
 0.6085830042105451,
 0.4570387283628752,
 0.6674701982444136,
 0.5667086409016293,
 0.710110955902248,
 0.48789129723115376,
 0.6657624958397428,
 0.6089284148932655,
 0.6397685755956675,
 0.6410709224098974,
 0.6210283939684125,
 0.7366931810218385,
 0.6403435054918312,
 0.5492294440786976,
 0.49865430736412675,
 0.6732835680238167,
 0.7035631613026931,
 0.7254576384148604,
 0.6010349558544669]